[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Relationships &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: the classes with their relationships, `SessionLocal` and `plan_summer`, which it runs
to open Summer 2026. Run it first. The tasks do not depend on one another, and the last cell removes
the scratch folder.


In [1]:
import logging
import shutil
import warnings
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, Date, ForeignKey, Integer, MetaData, String, Table, UniqueConstraint,
                        create_engine, event, func, insert, inspect, select)
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, sessionmaker
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def plan_summer(SessionLocal, offerings):
    """Open a Summer 2026 term with a section of every course in `offerings`, enrolling the students listed
    for each by email, all through relationships; return what the term holds, read back through them."""
    with SessionLocal.begin() as session:
        courses = {course.code: course for course in session.scalars(select(Course).where(Course.code.in_(offerings)))}
        students = {student.email: student for student in session.scalars(select(Student))}

        summer = Term(name="Summer 2026", starts_on=date(2026, 6, 1))
        session.add(summer)                                   # first, so whatever joins it joins the session
        for code, emails in offerings.items():
            section = Section(course=courses[code], capacity=15)
            summer.sections.append(section)
            for email in sorted(emails, key=lambda email: students[email].name):
                section.enrollments.append(Enrollment(student=students[email]))

        session.flush()
        return {section.course.code: (section.id, [enrollment.student.name for enrollment in section.enrollments])
                for section in summer.sections}


SessionLocal = sessionmaker(engine)
with SessionLocal.begin() as session:
    session.add(Student(name="Zoe Nakamura", email="znakamura@college.edu", program="Computer Science",
                        started_on=date(2026, 1, 12)))
print(plan_summer(SessionLocal, {"STA-200": ["areyes@college.edu", "bokafor@college.edu", "znakamura@college.edu"],
                                 "CSC-101": ["cmartin@college.edu", "znakamura@college.edu"]}))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}
{'STA-200': (41, ['Ana Reyes', 'Ben Okafor', 'Zoe Nakamura']), 'CSC-101': (42, ['Chloe Martin', 'Zoe Nakamura'])}


**1.** A course's sections.


In [2]:
with SessionLocal() as session:
    calculus = session.scalars(select(Course).where(Course.code == "MAT-120")).one()
    for section in calculus.sections:
        print(section, section.term.name)


Section(3) Fall 2024
Section(13) Spring 2025
Section(23) Fall 2025
Section(33) Spring 2026


`course.sections` is ordered by term, so the four sections come back from Fall 2024 to Spring 2026.


**2.** A section's students.


In [3]:
with SessionLocal() as session:
    data_structures = session.get(Section, 36)
    print(data_structures.course.title, "|", [enrollment.student.name for enrollment in data_structures.enrollments])


Data Structures | ['Ben Okafor', 'Felix Wagner', 'Isabel Costa', 'Liam Murphy', 'Pavel Novak', 'Sam Ito', 'Vera Kowalski']


**3.** One append, both sides.


In [4]:
with SessionLocal() as session:
    grace = session.get(Student, 7)
    statistics = session.get(Section, 40)
    with session.no_autoflush:                       # nothing is flushed while the enrollment is half made
        added = Enrollment(section=statistics)
        grace.enrollments.append(added)
        print("in statistics.enrollments:", added in statistics.enrollments, "| student:", added.student)
        print("still pending, never flushed:", inspect(added).pending)
    session.rollback()


in statistics.enrollments: True | student: Student('Grace Lin', 'Computer Science')
still pending, never flushed: True


Appending to `grace.enrollments` set `added.student`, and giving the enrollment its section put it
in `statistics.enrollments`, both before any flush. Reading each collection for the first time
loaded it with a query, and without `no_autoflush` that query would have tried to flush an
enrollment that had a section and no student yet, the warning in the fourth of the Common errors.
The rollback discarded it.


**4.** A count per program, through a relationship.


In [5]:
PER_PROGRAM = (
    select(Student.program, func.count())
    .join(Student.enrollments)
    .join(Enrollment.section)
    .where(Section.term_id == 4)
    .group_by(Student.program)
    .order_by(Student.program)
)
with SessionLocal() as session:
    print(session.execute(PER_PROGRAM).all())


[('Biology', 15), ('Computer Science', 15), ('History', 15), ('Mathematics', 15), ('Psychology', 15)]


Fifteen for every program, since every student takes three courses in Spring 2026 and every program
has five students. Zoe Nakamura's Spring 2026 enrollments would count too, and in this notebook Zoe
has none.


**5.** The relationships of `Enrollment`.


In [6]:
for rel in inspect(Enrollment).relationships:
    print(rel.key, rel.direction.name, rel.mapper.class_.__name__)


student MANYTOONE Student
section MANYTOONE Section


**6.** A term's sections, and the students in them.


In [7]:
with SessionLocal() as session:
    summer = session.scalars(select(Term).where(Term.name == "Summer 2026")).one()
    counts = {section.course.code: len(section.enrollments) for section in summer.sections}
print(counts, "| the whole term:", sum(counts.values()))


{'CSC-101': 2, 'STA-200': 3} | the whole term: 5


`summer.sections` is ordered by course id, so Statistics, course 10, comes after Programming I,
course 5.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Relationships](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/11-relationships.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
